# 05 Stratified Analysis and Confounding — Exercises

Practice stratified analysis and the Mantel-Haenszel method using the Legionnaires' disease line list from the Pine and Cypress Nursing Home.

In [ ]:
# Google Colab setup -- skip this cell if running locally
import sys
import os
if 'google.colab' in sys.modules:
    !git clone https://github.com/ancientsky/python4epi.git /content/python4epi 2>/dev/null || true
    os.chdir('/content/python4epi')
    !pip install -q -e .

In [ ]:
import pathlib

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from scipy.stats import chi2_contingency
from epi_learning.metrics import risk_ratio

# -- CJK font setup (prevents Chinese labels from showing as boxes) --
# Scan system font directories and explicitly register CJK fonts (more reliable than relying on the cache)
for _font_dir in map(pathlib.Path, ["/usr/share/fonts", "/usr/local/share/fonts"]):
    if _font_dir.exists():
        for _fp in sorted(_font_dir.rglob("*")):
            if _fp.suffix.lower() in {".ttf", ".ttc", ".otf"} and (
                "CJK" in _fp.name or "WenQuanYi" in _fp.name or "wqy" in _fp.name
            ):
                try:
                    fm.fontManager.addfont(str(_fp))
                except Exception:
                    pass

plt.rcParams["font.sans-serif"] = [
    "Noto Sans CJK TC", "Noto Sans CJK SC", "Noto Sans CJK JP",
    "Noto Sans TC", "Microsoft JhengHei",
    "WenQuanYi Zen Hei", "SimHei", "Arial Unicode MS",
    "Heiti TC", "DejaVu Sans",
]
plt.rcParams["axes.unicode_minus"] = False
plt.style.use("ggplot")
plt.rcParams["figure.dpi"] = 150

df = pd.read_csv("data/synthetic/legionella_outbreak.csv")
df["infected"] = (df["clinical_severity"] != "not_ill").astype(int)

## Question 1: Confounding Analysis of Hydrotherapy Use

Ch03 found that `hydrotherapy_use` is also associated with infection. Now we suspect `functional_status` is likewise a confounder of it.

1. Compute the crude RR for `hydrotherapy_use → infected`
2. Verify the three confounder requirements: is `functional_status` associated with `hydrotherapy_use`? (use crosstab)
3. Stratify by `functional_status` and compute the RR and 95% CI in each stratum
4. Compare: the difference between the stratum-specific RRs and the crude RR

In [ ]:
# TODO: Crude RR (hydrotherapy_use -> infected)
# TODO: Verify the confounder conditions
# TODO: Stratum-specific RR

## Question 2: Mantel-Haenszel Adjustment

Continuing from Question 1, compute the MH adjusted RR for `hydrotherapy_use → infected` (controlling for `functional_status`).

1. Compute it by hand using the formula $RR_{MH} = \frac{\sum_i a_i(c_i+d_i)/N_i}{\sum_i c_i(a_i+b_i)/N_i}$
2. Compare the crude RR and the MH RR—is the difference large?
3. Conclusion: is functional status a confounder of hydrotherapy use?

In [ ]:
# TODO: Compute the MH adjusted RR
# TODO: Compare crude RR vs MH RR

## Question 3 (Challenge): Stratify by Age Group + Forest Plot

1. Create `age_group` (60-69 / 70-79 / 80-89 / 90+)
2. Stratify by `age_group` and compute the stratum-specific RR and 95% CI for `shower_use → infected`
3. Draw a forest plot, marking the crude RR with a red dashed line
4. Compute the MH adjusted RR
5. Is age a confounder of shower use? Is there interaction?

In [ ]:
# TODO: Create age_group
# TODO: Stratum-specific RR + 95% CI
# TODO: Forest plot
# TODO: MH adjusted RR
# TODO: Interpretation

## Question 4: Confounding of Comorbidity and Age on Mortality (COVID-19 Scenario)

A hospital's COVID-19 case line list records age group (`age_group`), presence of comorbidity (`comorbidity`, e.g. diabetes, cardiovascular disease) and death outcome (`death`). You suspect that age is a confounder of the effect of comorbidity on mortality.

1. Compute the crude RR for `comorbidity → death`
2. Verify the three confounder requirements: is `age_group` associated with both `comorbidity` and `death`? (use crosstab)
3. Stratify by `age_group` and compute the RR and 95% CI in each stratum
4. Compute the Mantel-Haenszel adjusted RR and compare it with the crude RR
5. Interpret: is age a confounder of comorbidity's effect on mortality? How does the adjusted conclusion differ from the crude analysis?

In [ ]:
# --- Data: age and comorbidity of COVID-19 cases ---
rng = np.random.default_rng(20)
n = 800

age_group = rng.choice(["under60", "60plus"], size=n, p=[0.6, 0.4])
comorbidity = np.array([
    rng.binomial(1, 0.5 if ag == "60plus" else 0.15) for ag in age_group
])
death_prob = np.select(
    [
        (age_group == "under60") & (comorbidity == 0),
        (age_group == "under60") & (comorbidity == 1),
        (age_group == "60plus") & (comorbidity == 0),
        (age_group == "60plus") & (comorbidity == 1),
    ],
    [0.02, 0.05, 0.12, 0.30],
)
death = rng.binomial(1, death_prob)

covid_df = pd.DataFrame({
    "case_id": [f"C{i:04d}" for i in range(n)],
    "age_group": age_group,
    "comorbidity": comorbidity,
    "death": death,
})

# TODO: Compute the crude RR for comorbidity -> death
# TODO: Verify the three confounder requirements: is age_group associated with both comorbidity and death?
# TODO: Stratify by age_group and compute the RR and 95% CI in each stratum
# TODO: Compute the Mantel-Haenszel adjusted RR and compare with the crude RR
# TODO: Interpret: is age a confounder of comorbidity's effect on mortality?

## Question 5: Confounding Analysis of Vaccination (Influenza Scenario)

Community influenza surveillance data records age group (`age_group`), influenza vaccination status (`vaccinated`), and influenza infection status (`infected`). Older adults are a high-risk group with a higher vaccination rate, but they also have a higher risk of infection — this is a classic example of "confounding by indication."

1. Compute the crude RR for `vaccinated → infected`
2. Verify the three confounder requirements: is `age_group` associated with both `vaccinated` and `infected`?
3. Stratify by `age_group` and compute the RR and 95% CI in each stratum
4. Compute the Mantel-Haenszel adjusted RR
5. Interpret: which is closer to the vaccine's "true" protective effect, the crude RR or the MH RR? Why does the crude analysis tend to underestimate vaccine effectiveness?

In [ ]:
# --- Data: influenza vaccination and age ---
rng = np.random.default_rng(11)
n = 800

age_group = rng.choice(["under65", "65plus"], size=n, p=[0.7, 0.3])
vaccinated = np.array([
    rng.binomial(1, 0.7 if ag == "65plus" else 0.3) for ag in age_group
])
infect_prob = np.select(
    [
        (age_group == "under65") & (vaccinated == 0),
        (age_group == "under65") & (vaccinated == 1),
        (age_group == "65plus") & (vaccinated == 0),
        (age_group == "65plus") & (vaccinated == 1),
    ],
    [0.20, 0.10, 0.40, 0.20],
)
infected = rng.binomial(1, infect_prob)

flu_df = pd.DataFrame({
    "case_id": [f"F{i:04d}" for i in range(n)],
    "age_group": age_group,
    "vaccinated": vaccinated,
    "infected": infected,
})

# TODO: Compute the crude RR for vaccinated -> infected
# TODO: Verify the three confounder requirements: is age_group associated with both vaccinated and infected?
# TODO: Stratify by age_group and compute the RR and 95% CI in each stratum
# TODO: Compute the Mantel-Haenszel adjusted RR
# TODO: Interpret: does the crude RR underestimate the vaccine's protective effect?

## Question 6: Confounding Analysis of Food Exposure (Hepatitis A Scenario)

A hepatitis A cluster broke out after a group meal. The line list records prior hepatitis A vaccination (`vaccinated`), whether shellfish were eaten (`ate_shellfish`), and infection status (`infected`). You suspect that vaccination history confounds the association between food exposure and infection.

1. Compute the crude RR for `ate_shellfish → infected`
2. Verify the three confounder requirements: is `vaccinated` associated with both `ate_shellfish` and `infected`?
3. Stratify by `vaccinated` and compute the RR and 95% CI in each stratum
4. Compute the Mantel-Haenszel adjusted RR and compare it with the crude RR
5. Interpret: is vaccination history a confounder of the association between shellfish consumption and infection?

In [ ]:
# --- Data: hepatitis A cluster at a group meal ---
rng = np.random.default_rng(5)
n = 900

vaccinated = rng.binomial(1, 0.35, size=n)
ate_shellfish = np.array([
    rng.binomial(1, 0.3 if v == 1 else 0.6) for v in vaccinated
])
infect_prob = np.select(
    [
        (vaccinated == 0) & (ate_shellfish == 0),
        (vaccinated == 0) & (ate_shellfish == 1),
        (vaccinated == 1) & (ate_shellfish == 0),
        (vaccinated == 1) & (ate_shellfish == 1),
    ],
    [0.05, 0.35, 0.01, 0.07],
)
infected = rng.binomial(1, infect_prob)

hav_df = pd.DataFrame({
    "case_id": [f"H{i:04d}" for i in range(n)],
    "vaccinated": vaccinated,
    "ate_shellfish": ate_shellfish,
    "infected": infected,
})

# TODO: Compute the crude RR for ate_shellfish -> infected
# TODO: Verify the three confounder requirements: is vaccinated associated with both ate_shellfish and infected?
# TODO: Stratify by vaccinated and compute the RR and 95% CI in each stratum
# TODO: Compute the Mantel-Haenszel adjusted RR and compare with the crude RR
# TODO: Interpret: is vaccination history a confounder of the food exposure association?

## Question 7: Stratified Analysis by Region (Dengue Fever Scenario)

A county-wide dengue fever outbreak investigation's line list records residential region (`region`: urban / suburban / rural), presence of standing water containers at home (`standing_water`), and dengue infection status (`infected`). Vector mosquito density and residents' standing-water management habits differ across regions, so you suspect region is a confounder.

1. Compute the crude RR for `standing_water → infected`
2. Verify the three confounder requirements: is `region` associated with both `standing_water` and `infected`?
3. Stratify by `region` and compute the RR and 95% CI in each stratum
4. Compute the Mantel-Haenszel adjusted RR
5. Interpret: is region a confounder of the association between standing water exposure and dengue infection?

In [ ]:
# --- Data: cross-regional dengue fever investigation ---
rng = np.random.default_rng(42)
n = 900

region = rng.choice(["urban", "suburban", "rural"], size=n, p=[0.4, 0.35, 0.25])
water_p = {"urban": 0.2, "suburban": 0.4, "rural": 0.6}
standing_water = np.array([rng.binomial(1, water_p[r]) for r in region])
infect_p = {
    ("urban", 0): 0.03, ("urban", 1): 0.09,
    ("suburban", 0): 0.08, ("suburban", 1): 0.24,
    ("rural", 0): 0.15, ("rural", 1): 0.45,
}
infect_prob = np.array([infect_p[(r, w)] for r, w in zip(region, standing_water)])
infected = rng.binomial(1, infect_prob)

dengue_df = pd.DataFrame({
    "case_id": [f"D{i:04d}" for i in range(n)],
    "region": region,
    "standing_water": standing_water,
    "infected": infected,
})

# TODO: Compute the crude RR for standing_water -> infected
# TODO: Verify the three confounder requirements: is region associated with both standing_water and infected?
# TODO: Stratify by region (urban / suburban / rural) and compute the RR and 95% CI in each stratum
# TODO: Compute the Mantel-Haenszel adjusted RR
# TODO: Interpret: is region a confounder of the standing water exposure and dengue infection association?

## Question 8 (Challenge): Comorbidity Confounding Analysis (Tuberculosis Scenario)

A tuberculosis contact-screening dataset's line list records presence of diabetes (`diabetes`), close contact with a TB case (`close_contact`), and confirmed active tuberculosis (`active_tb`). Diabetes weakens the immune system and increases the risk of TB infection; at the same time, diabetic patients' care and living arrangements may also affect their contact history, so you suspect diabetes is a confounder.

1. Compute the crude OR for `close_contact → active_tb`
2. Verify the three confounder requirements: is `diabetes` associated with both `close_contact` and `active_tb`?
3. Stratify by `diabetes` and compute the OR and 95% CI in each stratum
4. Draw a forest plot, marking the crude OR with a red dashed line
5. Compute the Mantel-Haenszel adjusted OR
6. Interpret: is diabetes a confounder of the association between close contact and active tuberculosis, or an effect modifier? Are the stratum-specific ORs similar?

In [ ]:
# --- Data: tuberculosis contact screening ---
from epi_learning.metrics import odds_ratio

rng = np.random.default_rng(291)
n = 900

diabetes = rng.binomial(1, 0.25, size=n)
close_contact = np.array([
    rng.binomial(1, 0.5 if d == 1 else 0.25) for d in diabetes
])
tb_prob = np.select(
    [
        (diabetes == 0) & (close_contact == 0),
        (diabetes == 0) & (close_contact == 1),
        (diabetes == 1) & (close_contact == 0),
        (diabetes == 1) & (close_contact == 1),
    ],
    [0.02, 0.10, 0.08, 0.32],
)
active_tb = rng.binomial(1, tb_prob)

tb_df = pd.DataFrame({
    "case_id": [f"T{i:04d}" for i in range(n)],
    "diabetes": diabetes,
    "close_contact": close_contact,
    "active_tb": active_tb,
})

# TODO: Compute the crude OR for close_contact -> active_tb
# TODO: Verify the three confounder requirements: is diabetes associated with both close_contact and active_tb?
# TODO: Stratify by diabetes and compute the OR and 95% CI in each stratum
# TODO: Draw a forest plot, marking the crude OR with a red dashed line
# TODO: Compute the Mantel-Haenszel adjusted OR
# TODO: Interpret: is diabetes a confounder or an effect modifier? Are the stratum-specific ORs similar?